In [ ]:
import os

paths_to_check = [
    "/content/asl_dataset/WLASL_v0.3.json",
    "/content/asl_dataset/videos",
    "/content/asl_dataset/expanded_sample",
    "/content/asl_outputs/matched_video_inventory.csv",
    "/content/asl_outputs/landmark_features.npy",
    "/content/asl_outputs/processed_manifest.csv",
    "/content/asl_outputs/asl_baseline_predictions.csv",
    "/content/asl_outputs/asl_baseline_model.keras",
    "/content/asl_outputs/asl_label_mapping.json",
    "/content/asl_models",
    "/content/drive/MyDrive/ASL_Citizen_Project",
]

for path in paths_to_check:
    print(f"\n{path}")

    if os.path.isfile(path):
        print("  FILE EXISTS | Size:", round(os.path.getsize(path) / 1024, 2), "KB")
    elif os.path.isdir(path):
        items = os.listdir(path)
        print("  FOLDER EXISTS | Items:", len(items))
        print("  Sample:", items[:8])
    else:
        print("  NOT FOUND in current runtime")

In [ ]:
import os
import json
import pandas as pd
from google.colab import drive

# Reconnect Google Drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

# Inspect matched inventory
inventory_path = "/content/asl_outputs/matched_video_inventory.csv"

if os.path.exists(inventory_path):
    df = pd.read_csv(inventory_path)
    print("INVENTORY SHAPE:", df.shape)
    print("COLUMNS:", df.columns.tolist())
    display(df.head())

# Inspect predictions
pred_path = "/content/asl_outputs/asl_baseline_predictions.csv"

if os.path.exists(pred_path):
    print("\nBASELINE PREDICTIONS:")
    display(pd.read_csv(pred_path).head(10))

# Inspect label mapping
mapping_path = "/content/asl_outputs/asl_label_mapping.json"

if os.path.exists(mapping_path):
    with open(mapping_path, "r") as f:
        print("\nLABEL MAPPING:")
        print(json.load(f))

# Search for saved feature arrays and manifests
print("\nSEARCHING /content FOR RECOVERABLE FILES:")

extensions = (".npy", ".npz", ".csv", ".keras", ".h5", ".json")

for root, dirs, files in os.walk("/content"):
    dirs[:] = [
        d for d in dirs
        if d not in {"node_modules", ".cache", "site-packages"}
    ]

    for filename in files:
        if filename.lower().endswith(extensions):
            path = os.path.join(root, filename)

            if any(term in filename.lower() for term in [
                "feature", "manifest", "landmark", "baseline",
                "prediction", "label", "inventory", "model"
            ]):
                print(path, "|", round(os.path.getsize(path) / 1024, 1), "KB")

# Check the expected Drive project folder
drive_path = "/content/drive/MyDrive/ASL_Citizen_Project"

print("\nASL CITIZEN DRIVE FOLDER:")
if os.path.isdir(drive_path):
    for root, dirs, files in os.walk(drive_path):
        print(root, "| folders:", dirs[:5], "| files:", files[:10])
        dirs[:] = dirs[:5]
else:
    print("Folder not found at the expected path.")
    print("Drive is mounted, but the project may be stored elsewhere.")

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

BATCH_DIR = "/content/drive/MyDrive/ASL_Citizen_Project/batches"

batch_files = sorted(glob.glob(os.path.join(BATCH_DIR, "batch_*.npz")))
failure_files = sorted(glob.glob(os.path.join(BATCH_DIR, "failures_*.csv")))

print("Batch files found:", len(batch_files))
print("Failure reports found:", len(failure_files))

total_successful = 0
total_failed = 0
empty_batches = 0

for path in batch_files:
    try:
        with np.load(path, allow_pickle=False) as data:
            features = data["features"]
            metadata = data["metadata"]

            count = len(features)
            total_successful += count

            if count == 0:
                empty_batches += 1

            if count > 0 or path == batch_files[0]:
                print(
                    os.path.basename(path),
                    "| feature shape:", features.shape,
                    "| metadata count:", len(metadata)
                )

    except Exception as e:
        print("COULD NOT READ:", os.path.basename(path), "|", str(e)[:150])

for path in failure_files:
    try:
        failures = pd.read_csv(path)
        total_failed += len(failures)
    except Exception as e:
        print("Could not read failure report:", os.path.basename(path))

print("\n===== SUMMARY =====")
print("Successfully extracted videos:", total_successful)
print("Failed video records:", total_failed)
print("Empty batches:", empty_batches)
print("Total batch files:", len(batch_files))

if failure_files:
    print("\nSample extraction errors:")
    first_failures = pd.read_csv(failure_files[0])
    display(first_failures.head(10))

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

BATCH_DIR = "/content/drive/MyDrive/ASL_Citizen_Project/batches"
MANIFEST_PATH = "/content/asl_100_class_manifest.csv"

batch_files = sorted(glob.glob(os.path.join(BATCH_DIR, "batch_*.npz")))

# Safely load the manifest if it exists
if os.path.exists(MANIFEST_PATH):
    manifest = pd.read_csv(MANIFEST_PATH)
    expected_files = set(manifest["Video file"].astype(str))
    print(f"Loaded manifest with {len(expected_files)} expected videos.")
else:
    print(f"Warning: Manifest not found at {MANIFEST_PATH}. Skipping comparison checks.")
    manifest = None
    expected_files = set()

total_features = 0
all_metadata = []

for path in batch_files:
    with np.load(path, allow_pickle=False) as data:
        features = data["features"]
        metadata = data["metadata"]

        total_features += len(features)
        all_metadata.extend(metadata.tolist())

metadata_df = pd.DataFrame([
    __import__("json").loads(item) for item in all_metadata
])

processed_files = set(metadata_df["Video file"].astype(str))

print("Saved feature records:", total_features)
print("Unique processed videos:", len(processed_files))
print("Duplicate feature records:", metadata_df["Video file"].duplicated().sum())

print("\nProcessed videos by split:")
print(metadata_df["split"].value_counts())

print("\nProcessed classes:", metadata_df["Gloss"].nunique())

if manifest is not None:
    print("\n===== COMPARISON WITH MANIFEST =====")
    print("Manifest videos:", len(expected_files))
    print("Missing videos:", len(expected_files - processed_files))
    print("Unexpected videos:", len(processed_files - expected_files))

    if expected_files == processed_files and metadata_df["Video file"].duplicated().sum() == 0:
        print("\nPASS: All manifest videos have exactly one feature record.")
    else:
        print("\nCHECK REQUIRED: Some videos are missing, duplicated, or unexpected.")

In [ ]:
import os
import glob

print("Checking possible manifest locations...\n")

paths = [
    "/content/asl_100_class_manifest.csv",
    "/content/drive/MyDrive/asl_100_class_manifest.csv",
    "/content/drive/MyDrive/ASL_Citizen_Project/asl_100_class_manifest.csv",
]

for path in paths:
    print(f"{'FOUND' if os.path.exists(path) else 'NOT FOUND'}: {path}")

print("\nSearching project folder...")
for path in glob.glob(
    "/content/drive/MyDrive/ASL_Citizen_Project/**/*.csv",
    recursive=True
):
    print(path)

In [ ]:
import os
import glob

print("Checking original ASL Citizen split files...\n")

search_paths = [
    "/content/ASL_Citizen/splits",
    "/content/drive/MyDrive/ASL_Citizen/splits",
    "/content"
]

for folder in search_paths:
    if os.path.isdir(folder):
        for name in ["train.csv", "val.csv", "test.csv"]:
            path = os.path.join(folder, name)
            if os.path.exists(path):
                print("FOUND:", path)

print("\nChecking dataset ZIP...")
for path in [
    "/content/ASL_Citizen.zip",
    "/content/drive/MyDrive/ASL_Citizen.zip"
]:
    print(("FOUND: " if os.path.exists(path) else "NOT FOUND: ") + path)

print("\nChecking project files...")
for path in glob.glob(
    "/content/drive/MyDrive/ASL_Citizen_Project/**/*",
    recursive=True
):
    if os.path.isfile(path) and not "/batches/" in path:
        print(path)

In [ ]:
import os

roots = [
    "/content",
    "/content/drive/MyDrive"
]

for root in roots:
    print(f"\nSearching: {root}")

    for current, dirs, files in os.walk(root):
        # Avoid wasting time inside the thousands of feature files
        dirs[:] = [
            d for d in dirs
            if d not in {"batches", ".cache", "node_modules"}
        ]

        for filename in files:
            lower = filename.lower()

            if (
                lower in {"train.csv", "val.csv", "test.csv",
                          "asl_100_class_manifest.csv"}
                or lower == "asl_citizen.zip"
            ):
                print(os.path.join(current, filename))

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd

BATCH_DIR = "/content/drive/MyDrive/ASL_Citizen_Project/batches"

batch_files = sorted(
    glob.glob(os.path.join(BATCH_DIR, "batch_*.npz"))
)

records = []

for path in batch_files:
    with np.load(path, allow_pickle=False) as data:
        for item in data["metadata"]:
            records.append(json.loads(item))

df = pd.DataFrame(records)

print("Total records:", len(df))
print("Columns:", df.columns.tolist())

for col in ["Video file", "Gloss", "Participant ID", "ASL-LEX Code", "split"]:
    if col in df.columns:
        print(f"\n{col}:")
        print("Missing values:", df[col].isna().sum())
        print("Unique values:", df[col].nunique())
        print(df[col].head(5).tolist())

print("\nVideos per split:")
print(df["split"].value_counts())

print("\nTop 15 glosses:")
print(df["Gloss"].value_counts().head(15))

output_path = (
    "/content/drive/MyDrive/ASL_Citizen_Project/"
    "recovered_feature_manifest.csv"
)

df.to_csv(output_path, index=False)
print("\nRecovered metadata saved to:", output_path)

In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/ASL_Citizen_Project/recovered_feature_manifest.csv"
df = pd.read_csv(path)

split_signers = {
    split: set(group["Participant ID"].astype(str))
    for split, group in df.groupby("split")
}

for split, signers in split_signers.items():
    print(f"{split}: {len(signers)} unique participants")

print("\nTrain-validation overlap:",
      len(split_signers["train"] & split_signers["val"]))

print("Train-test overlap:",
      len(split_signers["train"] & split_signers["test"]))

print("Validation-test overlap:",
      len(split_signers["val"] & split_signers["test"]))

print("\nVideos per split:")
print(df["split"].value_counts())

print("\nClasses represented in each split:")
print(df.groupby("split")["Gloss"].nunique())

if (
    not (split_signers["train"] & split_signers["val"])
    and not (split_signers["train"] & split_signers["test"])
    and not (split_signers["val"] & split_signers["test"])
):
    print("\nPASS: No participant overlap in the saved metadata.")
else:
    print("\nWARNING: Participant overlap exists. Investigate before claiming signer independence.")

In [ ]:

import os
import glob
import json
import numpy as np
import pandas as pd

BATCH_DIR = "/content/drive/MyDrive/ASL_Citizen_Project/batches"

batch_files = sorted(
    glob.glob(os.path.join(BATCH_DIR, "batch_*.npz"))
)

all_features = []
all_metadata = []

for path in batch_files:
    with np.load(path, allow_pickle=False) as data:
        all_features.append(data["features"])
        all_metadata.extend(
            json.loads(item) for item in data["metadata"]
        )

X = np.concatenate(all_features, axis=0)
metadata = pd.DataFrame(all_metadata)

print("Feature array shape:", X.shape)
print("Metadata shape:", metadata.shape)
print("Feature data type:", X.dtype)
print("Contains NaN:", np.isnan(X).any())
print("Contains infinite values:", np.isinf(X).any())
print("Metadata rows match features:", len(metadata) == len(X))

assert len(metadata) == len(X)
assert X.shape == (3150, 8, 258)
assert not np.isnan(X).any()
assert not np.isinf(X).any()

print("\nFeature assembly successful.")


In [ ]:

from sklearn.preprocessing import LabelEncoder
import numpy as np

# Encode sign labels as numbers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(metadata["Gloss"].astype(str))

# Get the existing split assignments
train_idx = np.where(metadata["split"].values == "train")[0]
val_idx = np.where(metadata["split"].values == "val")[0]
test_idx = np.where(metadata["split"].values == "test")[0]

# Split features and labels
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)
print("Number of classes:", len(label_encoder.classes_))

print("\nUnique classes in each split:")
print("Train:", len(np.unique(y_train)))
print("Validation:", len(np.unique(y_val)))
print("Test:", len(np.unique(y_test)))

assert len(np.unique(y_train)) == 100
assert len(np.unique(y_val)) == 100
assert len(np.unique(y_test)) == 100

print("\nData preparation successful.")


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score

tf.keras.utils.set_random_seed(42)

model = models.Sequential([
    layers.Input(shape=(8, 258)),

    layers.Bidirectional(
        layers.LSTM(128, return_sequences=True)
    ),
    layers.Dropout(0.3),

    layers.Bidirectional(
        layers.LSTM(64)
    ),
    layers.Dropout(0.3),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(100, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)
import numpy as np

# Predict classes for unseen test participants
test_probabilities = model.predict(X_test, batch_size=32, verbose=1)
y_pred = np.argmax(test_probabilities, axis=1)

# Calculate evaluation metrics
test_accuracy = accuracy_score(y_test, y_pred)
test_macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Test Macro-F1: {test_macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        zero_division=0
    )
)


In [ ]:

import numpy as np

# Separate hand and pose features
hand_features = X[:, :, :126]
pose_features = X[:, :, 126:]

print("HAND LANDMARKS")
print("Overall standard deviation:", round(float(np.std(hand_features)), 5))
print("Percentage of zero values:",
      round(float(np.mean(hand_features == 0) * 100), 2), "%")

print("\nPOSE LANDMARKS")
print("Overall standard deviation:", round(float(np.std(pose_features)), 5))
print("Percentage of zero values:",
      round(float(np.mean(pose_features == 0) * 100), 2), "%")

# Check whether each video has any nonzero landmark information
hand_empty = np.all(hand_features == 0, axis=(1, 2))
pose_empty = np.all(pose_features == 0, axis=(1, 2))

print("\nVIDEOS WITH NO HAND FEATURES:",
      int(hand_empty.sum()), "/", len(X))
print("VIDEOS WITH NO POSE FEATURES:",
      int(pose_empty.sum()), "/", len(X))

# Inspect feature variation across videos
print("\nMean nonzero values per frame:")
print("Hand:", round(float(np.mean(np.count_nonzero(hand_features, axis=2))), 2))
print("Pose:", round(float(np.mean(np.count_nonzero(pose_features, axis=2))), 2))


In [ ]:

import numpy as np

hand = X[:, :, :126]

# Each hand has 63 values: 21 landmarks x 3 coordinates
left_hand = hand[:, :, :63]
right_hand = hand[:, :, 63:126]

def detection_stats(name, hand_data):
    # Count frames with at least one nonzero coordinate
    detected = np.any(hand_data != 0, axis=2)

    print(f"\n{name}")
    print("Detected frames:", int(detected.sum()), "/", detected.size)
    print("Detection rate:", round(float(detected.mean() * 100), 2), "%")
    print("Videos with zero detected frames:",
          int((~detected).all(axis=1).sum()))

detection_stats("Hand feature group 1", left_hand)
detection_stats("Hand feature group 2", right_hand)

# How many frames contain information from either hand?
any_hand = np.any(hand != 0, axis=2)

print("\nEither hand has features:")
print("Frames with hand features:", int(any_hand.sum()), "/", any_hand.size)
print("Frames with no hand features:", int((~any_hand).sum()))
print("Videos with no hand features:", int((~any_hand).all(axis=1).sum()))


In [ ]:

import numpy as np

hand = X[:, :, :126]

# Check whether the two 63-value groups are ever active independently
group1_active = np.any(hand[:, :, :63] != 0, axis=2)
group2_active = np.any(hand[:, :, 63:] != 0, axis=2)

print("Frames with group 1 only:",
      int(np.sum(group1_active & ~group2_active)))

print("Frames with group 2 only:",
      int(np.sum(~group1_active & group2_active)))

print("Frames with both groups:",
      int(np.sum(group1_active & group2_active)))

print("Frames with neither group:",
      int(np.sum(~group1_active & ~group2_active)))

# Check whether the second group is entirely zero across the dataset
print("\nGroup 1 nonzero values:", int(np.count_nonzero(hand[:, :, :63])))
print("Group 2 nonzero values:", int(np.count_nonzero(hand[:, :, 63:])))

# Inspect a few frames with group 2 active
indices = np.argwhere(group2_active)

print("\nFirst 10 (video index, frame index) with group 2 active:")
print(indices[:10])


In [ ]:

import inspect

print(inspect.getsource(detection_stats))


In [ ]:

# Show notebook functions whose names may relate to landmark extraction
for name, obj in list(globals().items()):
    if callable(obj) and any(
        word in name.lower()
        for word in ["extract", "landmark", "feature", "frame"]
    ):
        if hasattr(obj, "__code__"):
            print("\nFUNCTION:", name)
            print(inspect.getsource(obj))


In [ ]:

import inspect

keywords = ["hand_landmarker", "detect_for_video", "hand_landmarks",
            "pose_landmarks", "hand_features", "258", "126"]

for name, obj in list(globals().items()):
    if callable(obj) and hasattr(obj, "__code__"):
        try:
            source = inspect.getsource(obj)
            if any(word in source.lower() for word in keywords):
                print(f"\n{'='*60}")
                print("FUNCTION:", name)
                print(source[:7000])
        except (OSError, TypeError):
            pass


In [ ]:

import os
import glob
import json

# Search common Colab locations for your project notebook
for root in ["/content", "/content/drive/MyDrive"]:
    for path in glob.glob(root + "/**/*.ipynb", recursive=True):
        if "asl" in os.path.basename(path).lower():
            print(path)


In [ ]:

import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/ASL_Signer_Independent_Multimodal_Model.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "hand_landmarker",
    "hand_landmarks",
    "hand_features",
    "detect_for_video",
    "pose_landmarks",
    "np.concatenate",
    "258"
]

for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(word in source.lower() for word in keywords):
        print(f"\n{'='*60}")
        print("CELL:", i)
        print(source[:5000])


In [ ]:

import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/ASL_Signer_Independent_Multimodal_Model.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])
    lines = source.lower()

    # Find likely feature extraction or batch processing cells
    keywords = [
        "hand_task",
        "hand_model",
        "hand_result",
        "hand_landmarker",
        "handedness",
        "hand_world_landmarks",
        "pose_result",
        "batch_",
        "features.append",
        "np.zeros",
        "258"
    ]

    if any(k in lines for k in keywords):
        print(f"\n{'='*70}")
        print(f"CELL {i} | {len(source)} characters")
        print(source[:3500])


In [ ]:

import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/ASL_Signer_Independent_Multimodal_Model.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    if i >= 102:
        break

    source = "".join(cell.get("source", []))
    if not source.strip():
        continue

    first_lines = "\n".join(source.splitlines()[:4])

    print(f"\n--- CELL {i} [{cell['cell_type']}] ---")
    print(first_lines[:450])


In [ ]:

import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/ASL_Signer_Independent_Multimodal_Model.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i in range(85, 91):
    cell = nb["cells"][i]
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])
        print(f"\n{'='*60}\nCELL {i}\n{'='*60}")
        print(source)



In [ ]:

import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/ASL_Signer_Independent_Multimodal_Model.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "mediapipe", "hand_landmarker.task", "pose_landmarker.task",
    "HandLandmarker", "PoseLandmarker", "NormalizedLandmark",
    "hand_landmarks", "pose_landmarks", "258", "63"
]

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))

    matches = [word for word in keywords if word.lower() in source.lower()]

    if matches:
        print(f"\n{'='*60}")
        print(f"CELL {i} | Type: {cell['cell_type']}")
        print("Matched terms:", matches)
        print(source[:5000])
